# Joins — CDC Simulation on Customers

Generate a CDC batch (**6 inserts, 6 updates, 6 deletes**), apply `MERGE` to `silver.customers`, and verify each operation type.

**Note:** This modifies `silver.customers`. Re-run silver customers notebook to restore from bronze if needed.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.joins.cdc_customers as cdc_module
importlib.reload(cdc_module)

from src.joins.business_questions import SilverJoinTables
from src.joins.cdc_customers import (
    CdcBatchCounts,
    generate_cdc_batch,
    merge_sql_template,
    apply_customer_cdc_merge,
    verify_cdc_merge,
    run_customer_cdc_simulation,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

tables = SilverJoinTables()
counts = CdcBatchCounts(inserts=6, updates=6, deletes=6)
print("Loaded from:", cdc_module.__file__)
print("Customers before:", spark.table(tables.customers).count())

In [ ]:
cdc_batch, batch_meta = generate_cdc_batch(spark, tables.customers, counts)
display(cdc_batch.groupBy("cdc_operation").count())
display(cdc_batch.limit(10))

print("MERGE statement:")
print(merge_sql_template(tables.customers))

In [ ]:
row_count_before = spark.table(tables.customers).count()
apply_customer_cdc_merge(spark, cdc_batch, tables.customers)
verification = verify_cdc_merge(spark, tables.customers, batch_meta, row_count_before)
print(verification)

In [ ]:
import json

report = {
    "task": "cdc_customer_merge",
    "target_table": tables.customers,
    "cdc_batch": {
        "total_records": batch_meta["batch_size"],
        "inserts": batch_meta["inserts"],
        "updates": batch_meta["updates"],
        "deletes": batch_meta["deletes"],
    },
    "merge_sql": merge_sql_template(tables.customers),
    "verification": verification,
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/joins_cdc_customers.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)